# 📈 Chương 4 — Dataset 2: Google Stock Price — Conv1D + LSTM
## Kỹ thuật: Conv1D (trích xuất đặc trưng cục bộ) + LSTM (xu hướng dài hạn)

**Pipeline:** EDA → Technical Indicators → Sliding Window → Conv1D+LSTM → Train → Forecast

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import glob, os, warnings

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11,
                     'axes.titlesize': 13, 'axes.titleweight': 'bold'})

DATA_DIR   = '../data/google_stock'
SAVE_DIR   = '../results/google_stock'
os.makedirs(SAVE_DIR, exist_ok=True)

LOOK_BACK  = 60   # 60 ngày
EPOCHS     = 50
BATCH_SIZE = 32

print(f'TF: {tf.__version__}')

## 📂 1. Load Data

In [ ]:
csv_files = glob.glob(f'{DATA_DIR}/**/*.csv', recursive=True) + glob.glob(f'{DATA_DIR}/*.csv')
print('Found:', csv_files)
df = pd.read_csv(csv_files[0])
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# Detect date and price columns
date_col  = [c for c in df.columns if 'date' in c.lower()]
close_col = [c for c in df.columns if 'close' in c.lower() or 'price' in c.lower()]
open_col  = [c for c in df.columns if 'open'  in c.lower()]
high_col  = [c for c in df.columns if 'high'  in c.lower()]
low_col   = [c for c in df.columns if 'low'   in c.lower()]
vol_col   = [c for c in df.columns if 'vol'   in c.lower()]

close_col = close_col[0] if close_col else df.columns[1]
print(f'Target (Close): {close_col}')

if date_col:
    df[date_col[0]] = pd.to_datetime(df[date_col[0]])
    df = df.sort_values(date_col[0]).reset_index(drop=True)

# Clean numeric columns
for col in df.columns:
    if df[col].dtype == object:
        df[col] = df[col].str.replace(',','').str.replace('$','')
        try: df[col] = pd.to_numeric(df[col])
        except: pass

df = df.dropna(subset=[close_col])
print(f'Final shape: {df.shape}')

## 📊 2. EDA + Technical Indicators

In [ ]:
prices = df[close_col].values.astype(float)

# Compute Moving Averages
ma_7   = pd.Series(prices).rolling(7).mean().values
ma_30  = pd.Series(prices).rolling(30).mean().values
ma_60  = pd.Series(prices).rolling(60).mean().values

# Bollinger Bands
roll_mean = pd.Series(prices).rolling(20).mean()
roll_std  = pd.Series(prices).rolling(20).std()
bb_upper  = (roll_mean + 2 * roll_std).values
bb_lower  = (roll_mean - 2 * roll_std).values

fig, axes = plt.subplots(2, 1, figsize=(15, 9))

# Price + Moving Averages
axes[0].plot(prices, color='steelblue', lw=1, alpha=0.8, label='Close Price')
axes[0].plot(ma_7,  color='#FF9800', lw=1.5, linestyle='--', label='MA-7')
axes[0].plot(ma_30, color='#4CAF50', lw=1.5, linestyle='--', label='MA-30')
axes[0].plot(ma_60, color='#E91E63', lw=1.5, linestyle='--', label='MA-60')
axes[0].fill_between(range(len(prices)), bb_upper, bb_lower, alpha=0.1, color='gray', label='Bollinger Bands')
axes[0].set_title('Google Stock Price + Moving Averages + Bollinger Bands')
axes[0].set_ylabel('Price ($)')
axes[0].legend(ncol=3, fontsize=9)

# Daily returns
returns = pd.Series(prices).pct_change().values * 100
axes[1].bar(range(len(returns)), returns, color=np.where(returns>=0, '#4CAF50', '#EF5350'),
            width=1, alpha=0.8)
axes[1].axhline(0, color='black', lw=1)
axes[1].set_title('Daily Returns (%)')
axes[1].set_xlabel('Time Step')
axes[1].set_ylabel('Return (%)')

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/01_stock_eda.png', bbox_inches='tight')
plt.show()

print(f'Price range: ${prices.min():.2f} — ${prices.max():.2f}')
print(f'Mean return: {np.nanmean(returns):.4f}%, Std: {np.nanstd(returns):.4f}%')

In [ ]:
# Return distribution + Volatility
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(returns[~np.isnan(returns)], bins=60, color='steelblue',
             edgecolor='white', alpha=0.85)
axes[0].axvline(0, color='red', linestyle='--', lw=2)
axes[0].set_title('Phân phối Daily Returns')
axes[0].set_xlabel('Return (%)')

# Rolling volatility (30-day)
vol = pd.Series(returns).rolling(30).std().values
axes[1].plot(vol, color='#E91E63', lw=1.5)
axes[1].fill_between(range(len(vol)), vol, alpha=0.2, color='#E91E63')
axes[1].set_title('Rolling Volatility (30-day Std of Returns)')
axes[1].set_xlabel('Time Step')
axes[1].set_ylabel('Volatility (%)')

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/02_returns_volatility.png', bbox_inches='tight')
plt.show()

## ⚙️ 3. Preprocessing — Sliding Window

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
prices_scaled = scaler.fit_transform(prices.reshape(-1, 1)).flatten()

def create_dataset(data, look_back):
    X, y = [], []
    for i in range(len(data) - look_back):
        X.append(data[i:i+look_back])
        y.append(data[i+look_back])
    return np.array(X), np.array(y)

X, y = create_dataset(prices_scaled, LOOK_BACK)
X = X.reshape(X.shape[0], X.shape[1], 1)  # (samples, timesteps, features)

split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f'X shape: {X.shape}  → (samples, look_back={LOOK_BACK}, 1)')
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 🏗️ 4. Conv1D + LSTM Model

In [ ]:
model = models.Sequential([
    # Conv1D: trích xuất đặc trưng cục bộ (local patterns)
    layers.Conv1D(filters=64, kernel_size=3, activation='relu',
                  input_shape=(LOOK_BACK, 1), padding='same', name='conv1d_1'),
    layers.Conv1D(filters=64, kernel_size=3, activation='relu',
                  padding='same', name='conv1d_2'),
    layers.MaxPooling1D(pool_size=2),
    layers.Dropout(0.2),
    # LSTM: học xu hướng dài hạn từ đặc trưng đã trích xuất
    layers.LSTM(100, return_sequences=True, name='lstm_1'),
    layers.Dropout(0.3),
    layers.LSTM(50, name='lstm_2'),
    layers.Dropout(0.3),
    layers.Dense(25, activation='relu'),
    layers.Dense(1, name='forecast')
], name='Conv1D_LSTM_StockPrice')

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
model.summary()

## 🚀 5. Training

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=7, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-7, verbose=1)
    ],
    verbose=1
)

## 📈 6. Evaluation & Forecast Visualization

In [ ]:
# Predict and inverse transform
y_pred_scaled = model.predict(X_test, verbose=0).flatten()
y_pred_orig   = scaler.inverse_transform(y_pred_scaled.reshape(-1,1)).flatten()
y_test_orig   = scaler.inverse_transform(y_test.reshape(-1,1)).flatten()

train_pred_s  = model.predict(X_train, verbose=0).flatten()
train_pred    = scaler.inverse_transform(train_pred_s.reshape(-1,1)).flatten()
y_train_orig  = scaler.inverse_transform(y_train.reshape(-1,1)).flatten()

rmse = np.sqrt(mean_squared_error(y_test_orig, y_pred_orig))
mae  = mean_absolute_error(y_test_orig, y_pred_orig)
r2   = r2_score(y_test_orig, y_pred_orig)

print(f'RMSE : ${rmse:.4f}')
print(f'MAE  : ${mae:.4f}')
print(f'R²   : {r2:.4f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Training history - loss
eps = range(1, len(history.history['loss'])+1)
axes[0,0].plot(eps, history.history['loss'],     'o-', color='#2196F3', lw=2, ms=4, label='Train Loss')
axes[0,0].plot(eps, history.history['val_loss'], 's-', color='#FF5722', lw=2, ms=4, label='Val Loss')
axes[0,0].set_title('Training Loss (MSE)')
axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

# Full forecast view
full_prices = prices[LOOK_BACK:]
train_line  = np.full(len(full_prices), np.nan)
test_line   = np.full(len(full_prices), np.nan)
train_line[:split] = train_pred
test_line[split:]  = y_pred_orig

axes[0,1].plot(full_prices, color='steelblue', lw=1, alpha=0.7, label='Actual')
axes[0,1].plot(train_line, color='#4CAF50', lw=1.5, label='Train Prediction')
axes[0,1].plot(test_line,  color='#FF5722', lw=1.5, label='Test Prediction')
axes[0,1].axvline(split, color='black', linestyle='--', lw=1.5, label='Train/Test split')
axes[0,1].set_title(f'Stock Forecast — RMSE=${rmse:.2f}, R²={r2:.4f}')
axes[0,1].set_ylabel('Price ($)'); axes[0,1].legend(fontsize=9)

# Zoom into test period
axes[1,0].plot(y_test_orig, color='steelblue', lw=2, label='Actual')
axes[1,0].plot(y_pred_orig, color='#FF5722', lw=2, linestyle='--', label='Predicted')
axes[1,0].fill_between(range(len(y_test_orig)), y_test_orig, y_pred_orig,
                        alpha=0.2, color='gray')
axes[1,0].set_title('Test Period: Actual vs Predicted')
axes[1,0].set_ylabel('Price ($)'); axes[1,0].legend()

# Scatter
axes[1,1].scatter(y_test_orig, y_pred_orig, alpha=0.5, s=15, color='steelblue')
mn, mx = min(y_test_orig.min(), y_pred_orig.min()), max(y_test_orig.max(), y_pred_orig.max())
axes[1,1].plot([mn,mx],[mn,mx],'r--',lw=2, label='Perfect Forecast')
axes[1,1].set_xlabel('Actual ($)'); axes[1,1].set_ylabel('Predicted ($)')
axes[1,1].set_title(f'Scatter — R²={r2:.4f}')
axes[1,1].legend()

plt.suptitle('Conv1D + LSTM — Google Stock Price Forecast', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/03_forecast_results.png', bbox_inches='tight')
plt.show()

In [ ]:
with open(f'{SAVE_DIR}/report.txt', 'w') as f:
    f.write('Google Stock Price — Conv1D + LSTM\n' + '='*50 + '\n')
    f.write(f'LOOK_BACK : {LOOK_BACK} days\n')
    f.write(f'Params    : {model.count_params():,}\n\n')
    f.write(f'Test RMSE : ${rmse:.4f}\n')
    f.write(f'Test MAE  : ${mae:.4f}\n')
    f.write(f'Test R²   : {r2:.4f}\n')

print('✅ Google Stock Conv1D+LSTM Done! Saved to', SAVE_DIR)